# Kaismart Solutions S.A.S. — Extracción y EDA inicial

**Objetivo:** extraer las dos fuentes de información de Kaismart (sistema comercial en MySQL y sistema logístico en Excel), y realizar una exploración inicial de su estructura, calidad y contenido, siguiendo el enunciado de la evaluación (Partes 1 a 6).

- Fuente 1 — `df_ventas`: tabla `ventas` de la base de datos MySQL `clientes` (5.000 registros esperados).
- Fuente 2 — `df_logistica`: archivo `kaismart_eventos_logisticos.xlsx` (50.000 registros esperados).
- Llave de integración entre ambas fuentes: `pedido_id` (se usará más adelante, en el notebook de transformación Medallion).

> Nota de uso: si aún no cuenta con la base de datos MySQL ni el archivo Excel reales, ejecute `python scripts/generate_mock_data.py` desde la raíz del proyecto para generar datos simulados con problemas de calidad realistas y poder correr este notebook de punta a punta.

In [1]:
import sys
from pathlib import Path

# Permite importar el paquete kaismart_etl (carpeta src/) sin instalarlo
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from kaismart_etl import business_questions as bq
from kaismart_etl import quality, stats
from kaismart_etl.extract import extract_logistica, extract_ventas, mostrar_comprobacion

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## Parte 1. Extracción desde la base de datos (MySQL)

Se importan las librerías necesarias (`sqlalchemy`, `pymysql`, `pandas`), se crea la conexión con la base de datos `clientes`, se consulta **toda** la tabla `ventas` y se cierra la conexión correctamente (ver `kaismart_etl/db.py` y `kaismart_etl/extract.py`, que usan un bloque `try/finally` para garantizar el cierre incluso si ocurre un error).

In [2]:
# extract_ventas() abre la conexion, ejecuta "SELECT * FROM ventas" y CIERRA
# la conexion en un bloque finally (ver kaismart_etl/extract.py)
df_ventas = extract_ventas()

mostrar_comprobacion(df_ventas, "df_ventas")

2026-09-30 21:15:55 | INFO     | kaismart_etl.db | Creando engine MySQL -> host=107.180.112.11 db=clientes tabla=ventas
2026-09-30 21:15:57 | INFO     | kaismart_etl.extract | Consultando todos los registros de la tabla 'ventas'...
2026-09-30 21:15:58 | INFO     | kaismart_etl.extract | Extraccion de ventas completada: 5000 registros
2026-09-30 21:15:58 | INFO     | kaismart_etl.extract | Conexion a MySQL cerrada correctamente.



Comprobacion de extraccion: df_ventas
Shape: (5000, 17)

Columnas (17):
['id_venta', 'pedido_id', 'fecha_venta', 'id_cliente', 'ciudad', 'canal', 'id_tienda', 'categoria', 'producto', 'cantidad', 'precio_unitario', 'descuento_pct', 'valor_bruto', 'valor_descuento', 'valor_neto', 'medio_pago', 'calificacion_cliente']

Primeros 5 registros (head):
   id_venta  pedido_id         fecha_venta id_cliente       ciudad             canal id_tienda          categoria               producto  cantidad  \
0         1  PED000001 2026-01-18 22:06:15   CLI00011  Bogotá D.C.  Aplicación móvil       NaN           Deportes       Banda caminadora         1   
1         2  PED000002 2026-04-25 21:07:01   CLI00802         Cali        Página web       NaN           Deportes  Mancuernas ajustables         1   
2         3  PED000003 2026-03-28 02:39:43   CLI00255     Medellín        Página web       NaN         Tecnología     Tablet 10 pulgadas         1   
3         4  PED000004 2026-04-30 12:35:40   CLI003

## Parte 2. Extracción desde el archivo Excel

Se lee `kaismart_eventos_logisticos.xlsx` con `pandas.read_excel` (motor `openpyxl`) y se almacena en `df_logistica`.

In [3]:
df_logistica = extract_logistica()

mostrar_comprobacion(df_logistica, "df_logistica")

2026-09-30 21:16:31 | INFO     | kaismart_etl.extract | Leyendo archivo Excel: D:\Cursos\Phyton\kaismart-etl\data\raw\kaismart_eventos_logisticos (1).xlsx
2026-09-30 21:16:40 | INFO     | kaismart_etl.extract | Extraccion de eventos logisticos completada: 50000 registros



Comprobacion de extraccion: df_logistica
Shape: (50000, 14)

Columnas (14):
['evento_id', 'pedido_id', 'fecha_evento', 'estado_evento', 'centro_logistico', 'ciudad_destino', 'transportadora', 'numero_guia', 'fecha_prometida_entrega', 'tiempo_etapa_horas', 'costo_envio', 'incidencia', 'observacion', 'Unnamed: 13']

Primeros 5 registros (head):
   evento_id  pedido_id         fecha_evento              estado_evento centro_logistico ciudad_destino transportadora numero_guia fecha_prometida_entrega  \
0          1  PED000001  2026-01-18 22:06:15            Pedido recibido      CEDI Bogotá    Bogotá D.C.            NaN         NaN     2026-01-20 14:06:15   
1          2  PED000001  2026-01-18 22:36:53              Pago aprobado      CEDI Bogotá    Bogotá D.C.            NaN         NaN     2026-01-20 14:06:15   
2          3  PED000001  2026-01-18 23:31:22  Orden enviada a logística      CEDI Bogotá    Bogotá D.C.            NaN         NaN     2026-01-20 14:06:15   
3          4  PED00000

## Parte 3. Comprensión inicial de los datasets

**¿Qué representa una fila de `df_ventas`?** Cada fila es **una transacción de venta**: un cliente compra una cantidad de un producto de una categoría determinada, en una ciudad y a través de un canal (tienda física, web o app), en una fecha específica, con un precio, un descuento y una calificación opcional del cliente.

**¿Qué representa una fila de `df_logistica`?** Cada fila es **un evento del ciclo de vida logístico de un pedido** (por ejemplo: "Pedido Recibido", "Despachado", "Entregado"). Por lo tanto, **un mismo `pedido_id` tiene múltiples filas** en `df_logistica` (una por cada etapa), lo que explica por qué esta fuente tiene ~10 veces más registros (50.000) que `df_ventas` (5.000).

A continuación se determina, para cada dataset: número de registros y variables, nombre y tipo de cada variable, no nulos, e identificación heurística de identificadores / categóricas / numéricas / fechas.

In [5]:
print(f"df_ventas: {df_ventas.shape[0]} registros, {df_ventas.shape[1]} variables")
print(f"df_logistica: {df_logistica.shape[0]} registros, {df_logistica.shape[1]} variables\n")

resumen_ventas_estructura = quality.resumen_estructura(df_ventas, "df_ventas")
resumen_logistica_estructura = quality.resumen_estructura(df_logistica, "df_logistica")

print("Estructura df_ventas:")
display(resumen_ventas_estructura)
print("\nEstructura df_logistica:")
display(resumen_logistica_estructura)

df_ventas: 5000 registros, 17 variables
df_logistica: 50000 registros, 14 variables

Estructura df_ventas:


,dataset,columna,tipo_dato,no_nulos,nulos,pct_nulos,valores_unicos
0,df_ventas,id_venta,int64,5000,0,0.00,5000
1,df_ventas,pedido_id,str,5000,0,0.00,5000
2,df_ventas,fecha_venta,datetime64[us],5000,0,0.00,5000
3,df_ventas,id_cliente,str,5000,0,0.00,2292
4,df_ventas,ciudad,str,5000,0,0.00,6
5,df_ventas,canal,str,5000,0,0.00,3
6,df_ventas,id_tienda,str,1499,3501,70.02,10
7,df_ventas,categoria,str,5000,0,0.00,5
8,df_ventas,producto,str,5000,0,0.00,30
9,df_ventas,cantidad,int64,5000,0,0.00,4



Estructura df_logistica:


,dataset,columna,tipo_dato,no_nulos,nulos,pct_nulos,valores_unicos
0,df_logistica,evento_id,int64,50000,0,0.00,49900
1,df_logistica,pedido_id,str,50000,0,0.00,5000
2,df_logistica,fecha_evento,str,49935,65,0.13,49765
3,df_logistica,estado_evento,str,50000,0,0.00,10
4,df_logistica,centro_logistico,str,49910,90,0.18,6
5,df_logistica,ciudad_destino,str,49955,45,0.09,6
6,df_logistica,transportadora,str,19912,30088,60.18,5
7,df_logistica,numero_guia,str,19932,30068,60.14,4999
8,df_logistica,fecha_prometida_entrega,str,49945,55,0.11,4999
9,df_logistica,tiempo_etapa_horas,float64,45000,5000,10.00,1748


In [6]:
# Clasificacion heuristica de variables: identificadores / categoricas / numericas / fechas
clasificacion_ventas = quality.sugerir_tipos_variables(df_ventas)
clasificacion_logistica = quality.sugerir_tipos_variables(df_logistica)

print("df_ventas ->", clasificacion_ventas)
print("df_logistica ->", clasificacion_logistica)

# Rango de fechas disponible en cada fuente
if "fecha_venta" in df_ventas.columns:
    print("\nRango fechas df_ventas:", quality.rango_fechas(df_ventas, "fecha_venta"))
if "fecha_evento" in df_logistica.columns:
    print("Rango fechas df_logistica:", quality.rango_fechas(df_logistica, "fecha_evento"))

df_ventas -> {'identificadores': ['id_venta', 'pedido_id', 'id_cliente', 'id_tienda'], 'categoricas': ['ciudad', 'canal', 'categoria', 'producto', 'medio_pago'], 'numericas': ['cantidad', 'precio_unitario', 'descuento_pct', 'valor_bruto', 'valor_descuento', 'valor_neto', 'calificacion_cliente'], 'fechas': ['fecha_venta']}
df_logistica -> {'identificadores': ['evento_id', 'pedido_id', 'numero_guia'], 'categoricas': ['estado_evento', 'centro_logistico', 'ciudad_destino', 'transportadora', 'incidencia', 'observacion', 'Unnamed: 13'], 'numericas': ['tiempo_etapa_horas', 'costo_envio'], 'fechas': ['fecha_evento', 'fecha_prometida_entrega']}

Rango fechas df_ventas: (Timestamp('2026-01-01 09:08:59'), Timestamp('2026-06-30 20:42:35'))
Rango fechas df_logistica: (Timestamp('2026-01-01 09:08:59'), Timestamp('2026-07-03 10:34:14'))


**Otros aspectos de calidad observados (punto 10):**

- `ciudad`, `canal`, `categoria` (ventas) y `estado_evento`, `ciudad_destino`, `transportadora` (logística) son texto libre con **inconsistencias de mayúsculas/minúsculas, tildes y espacios** (p. ej. "bogota", "BOGOTÁ", "Bogotá D.C. ") en lugar de un catálogo controlado.
- `costo_envio` llega mezclado: algunos valores numéricos y otros como texto con formato moneda (`"$ 45.000"`), lo que impide operar la columna directamente.
- `id_venta`, `pedido_id` y `evento_id` deberían ser claves únicas, pero se detectan duplicados (ver Parte 4.3).
- `pedido_id` es la única llave común entre las dos fuentes; algunos registros la tienen nula, lo que impedirá integrarlos en la capa Gold.

## Parte 4. Perfil inicial de calidad del dato

### 4.1 Valores nulos

Se cuantifica cantidad y porcentaje de nulos por variable (sin eliminar ni imputar nada), ordenado de mayor a menor.

**Lectura esperada:**
- `calificacion_cliente` con alto % de nulos es **normal**: el cliente no está obligado a calificar la compra.
- `incidencia` con alto % de nulos en logística es **normal**: la mayoría de los eventos no tiene ninguna incidencia asociada.
- `valor_descuento` nulo también es esperable cuando no se aplicó descuento, aunque idealmente debería llegar como `0` desde el origen (posible problema de captura).
- Nulos en `pedido_id`, `fecha_venta`/`fecha_evento`, `precio_unitario` o `costo_envio` sí representan un **problema de calidad**, porque afectan la trazabilidad, el cálculo financiero o la integración entre fuentes.

In [8]:
print("Perfil de nulos - df_ventas")
display(quality.perfil_nulos(df_ventas))

print("\nPerfil de nulos - df_logistica")
display(quality.perfil_nulos(df_logistica))

Perfil de nulos - df_ventas


,columna,cantidad_nulos,pct_nulos
0,id_tienda,3501,70.02
1,calificacion_cliente,2172,43.44
2,id_venta,0,0.00
3,id_cliente,0,0.00
4,pedido_id,0,0.00
5,ciudad,0,0.00
6,canal,0,0.00
7,categoria,0,0.00
8,fecha_venta,0,0.00
9,producto,0,0.00



Perfil de nulos - df_logistica


,columna,cantidad_nulos,pct_nulos
0,Unnamed: 13,49999,100.00
1,incidencia,49489,98.98
2,observacion,48277,96.55
3,transportadora,30088,60.18
4,numero_guia,30068,60.14
5,tiempo_etapa_horas,5000,10.00
6,centro_logistico,90,0.18
7,fecha_evento,65,0.13
8,fecha_prometida_entrega,55,0.11
9,ciudad_destino,45,0.09


### 4.2 Valores únicos (cardinalidad)

Se usa `nunique()` para clasificar cada variable como de alta o baja cardinalidad, y se revisan específicamente `id_venta`, `pedido_id` y `evento_id` (deberían comportarse como identificadores, es decir, cardinalidad ≈ número de filas).

In [9]:
print("Cardinalidad - df_ventas")
display(quality.perfil_cardinalidad(df_ventas))

print("\nCardinalidad - df_logistica")
display(quality.perfil_cardinalidad(df_logistica))

print("\nAtencion a identificadores:")
for col in ("id_venta", "pedido_id"):
    if col in df_ventas.columns:
        print(f"  df_ventas.{col}: {df_ventas[col].nunique()} valores unicos / {len(df_ventas)} filas")
for col in ("evento_id", "pedido_id"):
    if col in df_logistica.columns:
        print(f"  df_logistica.{col}: {df_logistica[col].nunique()} valores unicos / {len(df_logistica)} filas")

Cardinalidad - df_ventas


,columna,valores_unicos,pct_valores_unicos,clasificacion
0,id_venta,5000,100.00,posible identificador (casi todos los valores ...
1,pedido_id,5000,100.00,posible identificador (casi todos los valores ...
2,fecha_venta,5000,100.00,posible identificador (casi todos los valores ...
3,id_cliente,2292,45.84,alta cardinalidad
4,valor_neto,1325,26.50,alta cardinalidad
5,valor_descuento,693,13.86,alta cardinalidad
6,valor_bruto,448,8.96,alta cardinalidad
7,precio_unitario,133,2.66,alta cardinalidad
8,producto,30,0.60,alta cardinalidad
9,id_tienda,10,0.20,baja cardinalidad



Cardinalidad - df_logistica


,columna,valores_unicos,pct_valores_unicos,clasificacion
0,evento_id,49900,99.80,posible identificador (casi todos los valores ...
1,fecha_evento,49765,99.53,posible identificador (casi todos los valores ...
2,pedido_id,5000,10.00,alta cardinalidad
3,numero_guia,4999,10.00,alta cardinalidad
4,fecha_prometida_entrega,4999,10.00,alta cardinalidad
5,tiempo_etapa_horas,1748,3.50,alta cardinalidad
6,costo_envio,12,0.02,baja cardinalidad
7,estado_evento,10,0.02,baja cardinalidad
8,observacion,9,0.02,baja cardinalidad
9,centro_logistico,6,0.01,baja cardinalidad



Atencion a identificadores:
  df_ventas.id_venta: 5000 valores unicos / 5000 filas
  df_ventas.pedido_id: 5000 valores unicos / 5000 filas
  df_logistica.evento_id: 49900 valores unicos / 50000 filas
  df_logistica.pedido_id: 5000 valores unicos / 50000 filas


### 4.3 Duplicados

Se cuantifican filas completamente duplicadas y también duplicados en las columnas que deberían ser identificadores únicos (`id_venta`, `pedido_id` en ventas; `evento_id` en logística). **No se elimina nada en esta parte**, solo se documenta.

In [10]:
print("Duplicados - df_ventas")
print(quality.perfil_duplicados(df_ventas, columnas_id=["id_venta", "pedido_id"]))

print("\nDuplicados - df_logistica")
print(quality.perfil_duplicados(df_logistica, columnas_id=["evento_id", "pedido_id"]))

Duplicados - df_ventas
{'filas_totales': 5000, 'filas_duplicadas_completas': 0, 'filas_duplicadas_completas_excedentes': 0, 'duplicados_en_id_venta': 0, 'valores_id_venta_repetidos': 0, 'duplicados_en_pedido_id': 0, 'valores_pedido_id_repetidos': 0}

Duplicados - df_logistica
{'filas_totales': 50000, 'filas_duplicadas_completas': 198, 'filas_duplicadas_completas_excedentes': 99, 'duplicados_en_evento_id': 200, 'valores_evento_id_repetidos': 100, 'duplicados_en_pedido_id': 50000, 'valores_pedido_id_repetidos': 5000}


**Interpretación importante:** en `df_logistica`, `pedido_id` se repite de forma **legítima y esperada** (cada pedido tiene varios eventos/etapas), por lo que no debe tratarse como un problema de calidad. El identificador que sí debe ser único en logística es `evento_id`. En `df_ventas`, tanto `id_venta` como `pedido_id` deberían ser únicos (una fila = una venta = un pedido); cualquier duplicado ahí sí es un error de origen a documentar y corregir en Silver.

### 4.4 Tipos de datos

Se revisa si el tipo detectado por pandas es coherente con el significado real de cada variable (fechas, identificadores, monetarias, numéricas, categóricas).

In [11]:
tipos_esperados_ventas = {
    "id_venta": "identificador",
    "pedido_id": "identificador",
    "cliente_id": "identificador",
    "fecha_venta": "fecha",
    "ciudad": "categorica",
    "canal": "categorica",
    "categoria": "categorica",
    "metodo_pago": "categorica",
    "cantidad": "numerica",
    "precio_unitario": "monetaria",
    "valor_bruto": "monetaria",
    "valor_descuento": "monetaria",
    "valor_neto": "monetaria",
    "calificacion_cliente": "numerica",
}
print("Revision de tipos - df_ventas")
display(quality.revisar_tipos(df_ventas, tipos_esperados_ventas))

tipos_esperados_logistica = {
    "evento_id": "identificador",
    "pedido_id": "identificador",
    "fecha_evento": "fecha",
    "estado_evento": "categorica",
    "ciudad_destino": "categorica",
    "transportadora": "categorica",
    "incidencia": "categorica",
    "tiempo_etapa_horas": "numerica",
    "costo_envio": "monetaria",
}
print("\nRevision de tipos - df_logistica")
display(quality.revisar_tipos(df_logistica, tipos_esperados_logistica))

Revision de tipos - df_ventas


,columna,tipo_esperado,dtype_detectado,requiere_revision,motivo
0,id_venta,identificador,int64,False,
1,pedido_id,identificador,str,False,
2,fecha_venta,fecha,datetime64[us],False,
3,ciudad,categorica,str,False,
4,canal,categorica,str,False,
5,categoria,categorica,str,False,
6,cantidad,numerica,int64,False,
7,precio_unitario,monetaria,float64,False,
8,valor_bruto,monetaria,float64,False,
9,valor_descuento,monetaria,float64,False,



Revision de tipos - df_logistica


,columna,tipo_esperado,dtype_detectado,requiere_revision,motivo
0,evento_id,identificador,int64,False,
1,pedido_id,identificador,str,False,
2,fecha_evento,fecha,str,True,se esperaba datetime pero llego como texto/objeto
3,estado_evento,categorica,str,False,
4,ciudad_destino,categorica,str,False,
5,transportadora,categorica,str,False,
6,incidencia,categorica,str,False,
7,tiempo_etapa_horas,numerica,float64,False,
8,costo_envio,monetaria,int64,False,


## Parte 5. Estadísticos descriptivos y resúmenes

Análisis descriptivo de cada fuente **por separado**, sin integrarlas todavía.

In [12]:
# 5.1 Variables numericas: count, mean, std, min, p25, mediana, p75, max
cols_numericas_ventas = ["cantidad", "precio_unitario", "valor_bruto", "valor_descuento", "valor_neto", "calificacion_cliente"]
print("Resumen numerico - df_ventas")
display(stats.resumen_numerico(df_ventas, cols_numericas_ventas))

cols_numericas_logistica = ["tiempo_etapa_horas", "costo_envio"]
print("\nResumen numerico - df_logistica")
display(stats.resumen_numerico(df_logistica, cols_numericas_logistica))

Resumen numerico - df_ventas


,count,mean,std,min,p25,mediana,p75,max
cantidad,5000.0,1.456200,0.743903,1.0,1.0,1.0,2.0,4.0
precio_unitario,5000.0,455950.160000,445808.472395,75900.0,142400.0,289900.0,685900.0,2309900.0
valor_bruto,5000.0,668288.120000,828903.828696,75900.0,188800.0,391800.0,807400.0,8799600.0
valor_descuento,5000.0,34593.104000,79018.313185,0.0,0.0,5495.0,35970.0,1671920.0
valor_neto,5000.0,633695.016000,788075.292481,63920.0,176310.0,367110.0,764915.0,8799600.0
calificacion_cliente,2828.0,4.094767,1.013632,1.0,4.0,4.0,5.0,5.0



Resumen numerico - df_logistica


,count,mean,std,min,p25,mediana,p75,max
tiempo_etapa_horas,45000.0,4.004721,3.460415,0.08,1.25,3.055,5.89,24.69
costo_envio,50000.0,13609.494400,3607.858756,8900.00,9900.00,13900.000,15900.00,20790.00


In [14]:
# 5.2 Variables categoricas + resumenes especificos solicitados para df_ventas
resultados_ventas = stats.resumen_ventas(df_ventas)
for titulo, tabla in resultados_ventas.items():
    print(f"\n--- {titulo} ---")
    display(tabla)


--- ventas_por_ciudad ---


,ciudad,conteo,pct
0,Bogotá D.C.,1653,33.06
1,Cali,942,18.84
2,Medellín,919,18.38
3,Barranquilla,534,10.68
4,Bucaramanga,513,10.26
5,Pereira,439,8.78



--- ventas_por_canal ---


,canal,conteo,pct
0,Página web,2020,40.40
1,Tienda física,1499,29.98
2,Aplicación móvil,1481,29.62



--- ventas_por_categoria ---


...

,count,mean,std,min,p25,mediana,p75,max
valor_neto,5000.0,633695.016,788075.292481,63920.0,176310.0,367110.0,764915.0,8799600.0



--- distribucion_calificacion_cliente ---


,calificacion_cliente
count,2828.000000
mean,4.094767
std,1.013632
min,1.000000
25%,4.000000
50%,4.000000
75%,5.000000
max,5.000000



--- conteo_calificacion_cliente ---


,conteo
calificacion_cliente,
1.0,86
2.0,143
3.0,386
4.0,1015
5.0,1198


In [15]:
# Resumenes especificos solicitados para df_logistica
resultados_logistica = stats.resumen_logistica(df_logistica)
for titulo, tabla in resultados_logistica.items():
    print(f"\n--- {titulo} ---")
    display(tabla)


--- eventos_por_estado ---


,estado_evento,conteo,pct
0,Listo para despacho,5006,10.01
1,Despachado,5005,10.01
2,Orden enviada a logística,5004,10.01
3,En alistamiento,5001,10.00
4,Pedido recibido,5000,10.00
5,Pago aprobado,4999,10.00
6,Empacado,4998,10.00
7,En tránsito,4998,10.00
8,Entregado,4998,10.00
9,En ruta de entrega,4991,9.98



--- eventos_por_ciudad_destino ---


,ciudad_destino,conteo,pct
0,Bogotá D.C.,16510,33.02
1,Cali,9412,18.82
2,Medellín,9186,18.37
3,Barranquilla,5331,10.66
4,Bucaramanga,5128,10.26
5,Pereira,4388,8.78
6,NaN,45,0.09



--- frecuencia_transportadoras ---


...

,incidencia,conteo,pct
0,NaN,49489,98.98
1,Error de clasificación,98,0.20
2,Cliente ausente,91,0.18
3,Novedad climática,91,0.18
4,Retraso por tráfico,81,0.16
5,Paquete reprogramado,76,0.15
6,Dirección incompleta,74,0.15



--- resumen_tiempo_etapa_horas ---


,count,mean,std,min,p25,mediana,p75,max
tiempo_etapa_horas,45000.0,4.004721,3.460415,0.08,1.25,3.055,5.89,24.69



--- resumen_costo_envio ---


,count,mean,std,min,p25,mediana,p75,max
costo_envio,50000.0,13609.4944,3607.858756,8900.0,9900.0,13900.0,15900.0,20790.0


## Parte 6. Preguntas de negocio (conocimiento previo)

10 preguntas resueltas con `pandas` directamente sobre los datos **originales** (sin integrar), usando las funciones de `kaismart_etl/business_questions.py`.

In [16]:
print("1. Valor neto total de ventas por ciudad")
print(bq.q1_ciudad_mayor_valor_neto(df_ventas))

print("\n2. Numero de ventas por canal")
print(bq.q2_canal_mas_ventas(df_ventas))

print("\n3. Cantidad total vendida por categoria")
print(bq.q3_categoria_mas_vendida_cantidad(df_ventas))

print("\n4. Top 10 clientes por valor neto acumulado")
print(bq.q4_top10_clientes_valor_neto(df_ventas))

print("\n5. Numero de ventas por mes")
print(bq.q5_ventas_por_mes(df_ventas))

1. Valor neto total de ventas por ciudad
ciudad
Bogotá D.C.     1.033662e+09
Cali            6.161808e+08
Medellín        5.876038e+08
Bucaramanga     3.478731e+08
Barranquilla    3.374398e+08
Pereira         2.457158e+08
Name: valor_neto, dtype: float64

2. Numero de ventas por canal
canal
Página web          2020
Tienda física       1499
Aplicación móvil    1481
Name: count, dtype: int64

3. Cantidad total vendida por categoria
categoria
Tecnología           2032
Electrodomésticos    1565
Hogar                1340
Deportes             1207
Oficina              1137
Name: cantidad, dtype: int64

4. Top 10 clientes por valor neto acumulado


KeyError: 'cliente_id'

In [20]:
print("6. Transportadora con mas incidencias reportadas")
print(bq.q6_transportadora_mas_incidencias(df_logistica))

print("\n7. Estado de evento logistico mas frecuente")
print(bq.q7_estado_evento_mas_frecuente(df_logistica))

print("\n8. Tiempo promedio de etapa (horas) por transportadora")
print(bq.q8_tiempo_promedio_por_transportadora(df_logistica))

print("\n9. Costo de envio promedio por ciudad de destino")
print(bq.q9_costo_envio_promedio_por_ciudad(df_logistica))

print("\n10. % de ventas con calificacion de cliente y promedio")
print(bq.q10_pct_ventas_con_calificacion(df_ventas))

6. Transportadora con mas incidencias reportadas
transportadora
Envia               111
Inter Rapidísimo    103
Coordinadora        100
TCC                 100
Servientrega         96
Name: count, dtype: int64

7. Estado de evento logistico mas frecuente
estado_evento
Listo para despacho          5006
Despachado                   5005
Orden enviada a logística    5004
En alistamiento              5001
Pedido recibido              5000
Pago aprobado                4999
Empacado                     4998
En tránsito                  4998
Entregado                    4998
En ruta de entrega           4991
Name: count, dtype: int64

8. Tiempo promedio de etapa (horas) por transportadora
transportadora
Inter Rapidísimo    6.58
Envia               6.47
Coordinadora        6.46
TCC                 6.46
Servientrega        6.45
Name: tiempo_etapa_horas, dtype: float64

9. Costo de envio promedio por ciudad de destino
ciudad_destino
Cali            13774.0
Pereira         13685.0
Bogotá D.C.    

## Cierre del EDA

Los problemas de calidad detectados y cuantificados en este notebook (nulos, duplicados, categorías inconsistentes, tipos de dato incorrectos, valores fuera de rango) se resuelven de forma explícita y documentada en el notebook [`02_transformacion_medallion.ipynb`](02_transformacion_medallion.ipynb), que implementa las capas **Bronze → Silver → Gold** y produce `df_ventas_transformado` y `df_logistica_transformado`.